© 2026 Mechanome Bio. Proprietary and confidential.

# Run Segger
docs: https://elihei2.github.io/segger_dev/ , repo: https://github.com/EliHei2/segger_dev (pinned commit e7d5c70)

- Current Segger needs Python >=3.11 (docs still say 3.10). Installed with `--no-deps` in setup.ipynb.
- Reads Xenium output files, not the sdata zarr. So transcripts and nucleus boundaries are taken from the sdata,
  saved as Xenium-style parquet files in `../inputs/Segger/<dataset>/xenium_files/`, then passed to Segger.
- No docker here; that's for the ECR stage later.
- Workflow follows the repo's `docs/notebooks/segger_tutorial.ipynb` and `scripts/1_train_5k.py` / `2_predict_5k.py`.

In [ ]:
import os
import sys

import geopandas
import pandas

sys.path.append("..")
from utils import seg_utils

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DATASET_NAME = "20240208_core_s0005_spt-xen_onco_pancreas"
METHOD = "segger"
BOUNDING_BOX_MICRONS = (4673, 1476, 5036, 1805)  # microns; None = whole slide

WORK_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "Segger" / DATASET_NAME
XENIUM_FILES_DIRECTORY = WORK_DIRECTORY / "xenium_files"   # Xenium-style parquet inputs
GRAPH_DATASET_DIRECTORY = WORK_DIRECTORY / "graph_dataset"  # Segger's tiled training data
MODELS_DIRECTORY = WORK_DIRECTORY / "models"
PREDICTION_DIRECTORY = WORK_DIRECTORY / "predictions"
XENIUM_FILES_DIRECTORY.mkdir(parents=True, exist_ok=True)

sdata = seg_utils.load_sdata(DATASET_NAME)

In [ ]:
seg_utils.preview_bounding_box(sdata, BOUNDING_BOX_MICRONS)

In [ ]:
sdata_crop = seg_utils.crop_sdata(sdata, BOUNDING_BOX_MICRONS)

## Xenium-style inputs
Column names Segger expects (from `segger/data/parquet/_settings/xenium.yaml`). All in microns.

In [ ]:
transcripts = seg_utils.get_transcripts(sdata_crop)
transcripts = transcripts.rename(columns={"x": "x_location", "y": "y_location", "z": "z_location"})
transcripts["feature_name"] = transcripts["feature_name"].astype(str)
transcripts[["x_location", "y_location", "z_location", "feature_name", "overlaps_nucleus",
             "transcript_id", "cell_id", "qv"]].to_parquet(XENIUM_FILES_DIRECTORY / "transcripts.parquet", index=False)

# nucleus polygons -> one row per vertex
nuclei = sdata_crop[seg_utils.NUCLEI_KEY]
vertices = nuclei.geometry.exterior.get_coordinates()
nucleus_vertices = pandas.DataFrame({
    "cell_id": vertices.index.astype(str),  # index of get_coordinates() = nucleus id
    "vertex_x": vertices["x"].values,
    "vertex_y": vertices["y"].values,
})
nucleus_vertices.to_parquet(XENIUM_FILES_DIRECTORY / "nucleus_boundaries.parquet", index=False)

## Build Segger's graph dataset
Parameters from the tutorial. `sample_type="xenium"` uses nucleus boundaries as the prior.

In [ ]:
from segger.data.parquet.sample import STSampleParquet

sample = STSampleParquet(base_dir=XENIUM_FILES_DIRECTORY, n_workers=4, sample_type="xenium")
sample.save(
    data_dir=GRAPH_DATASET_DIRECTORY,
    k_bd=3, dist_bd=15.0,
    k_tx=3, dist_tx=5.0,
    tile_width=120, tile_height=120,
    neg_sampling_ratio=5.0,
    frac=1.0,
    val_prob=0.1, test_prob=0.2,
)

## Train

In [ ]:
from lightning import Trainer
from lightning.pytorch.loggers import CSVLogger
from torch_geometric.nn import to_hetero
from segger.models.segger_model import Segger
from segger.training.segger_data_module import SeggerDataModule
from segger.training.train import LitSegger

data_module = SeggerDataModule(data_dir=GRAPH_DATASET_DIRECTORY, batch_size=2, num_workers=2)
data_module.setup()

model = Segger(
    num_tx_tokens=data_module.train[0].x_dict["tx"].shape[1],  # number of genes
    init_emb=8, hidden_channels=64, out_channels=16, heads=4, num_mid_layers=3,
)
model = to_hetero(model, (["tx", "bd"], [("tx", "belongs", "bd"), ("tx", "neighbors", "tx")]), aggr="sum")

# one forward pass to initialise the lazy layers
batch = data_module.train[0]
model.forward(batch.x_dict, batch.edge_index_dict)

trainer = Trainer(
    accelerator="cuda", strategy="auto", precision="16-mixed", devices=1,
    max_epochs=100,
    default_root_dir=MODELS_DIRECTORY, logger=CSVLogger(MODELS_DIRECTORY),
)
trainer.fit(model=LitSegger(model=model), datamodule=data_module)

## Predict
`save_cell_masks=True` also writes `segger_cell_boundaries.parquet` (polygons from the assigned transcripts, microns).

In [ ]:
from segger.prediction.predict_parquet import load_model, segment

checkpoint_directory = sorted(MODELS_DIRECTORY.glob("lightning_logs/version_*/checkpoints"))[-1]
trained_model = load_model(checkpoint_directory)

segment(
    trained_model, data_module,
    save_dir=PREDICTION_DIRECTORY,
    seg_tag=DATASET_NAME,
    transcript_file=XENIUM_FILES_DIRECTORY / "transcripts.parquet",
    receptive_field={"k_bd": 4, "dist_bd": 15, "k_tx": 5, "dist_tx": 3},
    score_cut=0.5,
    use_cc=False,
    knn_method="kd_tree",
    save_transcripts=True,
    save_anndata=True,
    save_cell_masks=True,
    min_transcripts=5,
    cell_id_col="segger_cell_id",
    verbose=True,
)

## Save

In [ ]:
boundaries_path = sorted(PREDICTION_DIRECTORY.glob("*/segger_cell_boundaries.parquet"))[-1]
polygons = geopandas.read_parquet(boundaries_path)
seg_utils.save_segmentation(sdata, polygons, METHOD, DATASET_NAME, BOUNDING_BOX_MICRONS)